# Best predictors based on calibration, permutation

## 0. Package loading and installation

In [1]:
# For Jupyter/Colab notebooks
%reset -f
import gc
gc.collect()

import numpy as np
import pandas as pd
from rpy2.robjects import r, pandas2ri

!pip install scikit-survival # Install scikit-survival if not already installed
!pip install miceforest --no-cache-dir

from sksurv.metrics import (
    concordance_index_ipcw,
    brier_score,
    integrated_brier_score
)
from sksurv.util import Surv

pandas2ri.activate()

#Glimpse function
def glimpse(df, max_width=80):
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    for col in df.columns:
        dtype = df[col].dtype
        preview = df[col].astype(str).head(5).tolist()
        preview_str = ", ".join(preview)
        if len(preview_str) > max_width:
            preview_str = preview_str[:max_width] + "..."
        print(f"{col:<30} {str(dtype):<15} {preview_str}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.0/300.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.8 MB/s eta 0:00:00
  Attempting uninstall: osqp
    Found existing installation: osqp 1.0.5
    Uninstalling osqp-1.0.5:
      Successfully uninstalled osqp-1.0.5


In [2]:
from google.colab import userdata

# Names of the objects (and secrets)
object_names = [
    "times_eval_death",
    "times_eval_readm"
]

for name in object_names:
    file_id = userdata.get(name)   # secret stored with this key
    if file_id is None:
        raise ValueError(f"No file_id found in userdata for key '{name}'")

    output_file = f"{name}.parquet"
    print(f"Downloading {name} -> {output_file}")
    !gdown --id {file_id} --output {output_file} --quiet

  # Names of the objects (and secrets)
object_names = [
    "X_reduced_list.npz.gpg"
]

for name in object_names:
    file_id = userdata.get(name)   # secret stored with this key
    if file_id is None:
        raise ValueError(f"No file_id found in userdata for key '{name}'")

    output_file = f"{name}"
    print(f"Downloading {name} -> {output_file}")
    !gdown --id {file_id} --output {output_file} --quiet

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(


In [3]:
from google.colab import userdata
import subprocess

passphrase = userdata.get('passphrase')

# For binary files, it's better to let gpg handle the output file directly
result = subprocess.run([
    'gpg', '--batch', '--pinentry-mode', 'loopback',
    '--passphrase', passphrase,
    '--output', 'X_reduced_list.npz',  # Specify output file
    '--decrypt', 'X_reduced_list.npz.gpg'
], capture_output=True, text=True)

if result.returncode == 0:
    print("✅ File decrypted successfully!")
    print("Decrypted file saved as: X_reduced_list.npz")
else:
    print(f"❌ Decryption error: {result.stderr}")

✅ File decrypted successfully!
Decrypted file saved as: X_reduced_list.npz


Load objects in Python

In [4]:
# ---- Time grids ----
times_eval_death = pd.read_parquet("times_eval_death.parquet")["time"].to_numpy()
times_eval_readm = pd.read_parquet("times_eval_readm.parquet")["time"].to_numpy()

print("times_eval_death shape:", times_eval_death.shape)
print("times_eval_readm shape:", times_eval_readm.shape)


times_eval_death shape: (49,)
times_eval_readm shape: (50,)


Added outcomes

In [5]:
import pandas as pd
import numpy as np
from google.colab import userdata

# ---- Download y_surv_readm & y_surv_death ----
for name in ["y_surv_readm", "y_surv_death"]:
    file_id = userdata.get(name)
    if file_id is None:
        raise ValueError(f"No file_id found in userdata for key '{name}'")
    !gdown --id {file_id} --output {name}.parquet --quiet
    print(f"Downloaded {name}.parquet")

# ---- Load and reconstruct Surv-like structured arrays ----
df_y_readm = pd.read_parquet("y_surv_readm.parquet")
y_surv_readm = np.array(
    list(zip(df_y_readm["event"].astype(bool),
             df_y_readm["time"].astype(float))),
    dtype=[("event", "?"), ("time", "<f8")]
)

df_y_death = pd.read_parquet("y_surv_death.parquet")
y_surv_death = np.array(
    list(zip(df_y_death["event"].astype(bool),
             df_y_death["time"].astype(float))),
    dtype=[("event", "?"), ("time", "<f8")]
)

print("y_surv_readm shape:", y_surv_readm.shape)
print("y_surv_death shape:", y_surv_death.shape)


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloaded y_surv_readm.parquet
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloaded y_surv_death.parquet
y_surv_readm shape: (88504,)
y_surv_death shape: (88504,)


In [7]:
import numpy as np
import pandas as pd

# Load the data
data = np.load("X_reduced_list.npz", allow_pickle=True)

# Reconstruct as DataFrames (you'll need the original column names)
# Replace with your actual column names
feature_columns = [f'feature_{i}' for i in range(data['X_reduced_list_1'].shape[1])]  # Adjust this!

X_reduced_list = [
    pd.DataFrame(data['X_reduced_list_1'], columns=feature_columns),
    pd.DataFrame(data['X_reduced_list_2'], columns=feature_columns),
    pd.DataFrame(data['X_reduced_list_3'], columns=feature_columns),
    pd.DataFrame(data['X_reduced_list_4'], columns=feature_columns),
    pd.DataFrame(data['X_reduced_list_5'], columns=feature_columns)
]

# 3. “Best predictors” (variable importance) based on calibration

To identify predictors that most influenced overall prediction error (calibration and discrimination combined), we computed permutation importance using the Integrated Brier Score (IBS). Within each imputed dataset, we ran k-fold cross-validation and fitted Coxnet models in the training folds. For each fold, we first obtained the out-of-sample IBS using the test data. We then permuted one predictor at a time in the test set, recomputed survival predictions and the IBS, and recorded the increase in IBS (worsening of prediction error). These IBS increases were pooled across folds and imputations, so that `mean_increase_ibs` summarized how much each predictor worsened out-of-sample IBS on average, while respecting both multiple imputation and cross-validation. Sorting predictors by `mean_increase_ibs` and selecting the top 20 yielded the most influential predictors from the standpoint of overall calibration-weighted predictive accuracy.

In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import brier_score, integrated_brier_score
from joblib import Parallel, delayed

def permutation_importance_ibs_cv_mi(
    X_list,
    y_surv,
    times_eval,
    alpha_idx=25,
    n_splits=5,
    n_repeats=3,
    random_state=2125,
    l1_ratio=0.9,
    alpha_min_ratio=0.01,
    n_alphas=50,
    max_iter=100000,
    n_jobs=-1,
):
    """
    Highly optimized version using risk scores and proper baseline survival access.
    """
    # Convert to NumPy arrays upfront
    feature_names = X_list[0].columns.tolist()
    n_features = len(feature_names)
    X_list = [X.values.astype(np.float64, copy=False) for X in X_list]
    times_eval = np.asarray(times_eval, dtype=np.float64)
    n_imputations = len(X_list)

    # Precompute CV splits once
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cv_splits = list(kf.split(np.arange(X_list[0].shape[0])))

    def compute_fold(d, fold_idx, train_idx, test_idx):
        X_imp = X_list[d]
        X_train = X_imp[train_idx]
        X_test = X_imp[test_idx]
        y_train = y_surv[train_idx]
        y_test = y_surv[test_idx]

        local_rng = np.random.RandomState(random_state + d * n_splits + fold_idx)

        # Fit Coxnet
        model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            alpha_min_ratio=alpha_min_ratio,
            n_alphas=n_alphas,
            normalize=True,
            fit_baseline_model=True,
            max_iter=max_iter,
            verbose=False,
        )
        model.fit(X_train, y_train)

        eff_alpha_idx = min(alpha_idx, len(model.alphas_) - 1)
        alpha_val = model.alphas_[eff_alpha_idx]

        # Access baseline survival - baseline_survival_ is a StepFunction
        baseline_estimator = model._baseline_models[eff_alpha_idx]
        S0 = baseline_estimator.baseline_survival_(times_eval)  # Evaluate at times_eval

        # Fast vectorized prediction using risk scores
        def get_survival_matrix(X):
            """Compute survival probabilities for all samples at all times."""
            risks = model.predict(X, alpha=alpha_val)
            # S(t|x) = S0(t)^exp(risk)
            return np.power(S0[None, :], np.exp(risks[:, None]))

        # Baseline predictions
        S_test = get_survival_matrix(X_test)

        # Get effective time grid
        times_brier, _ = brier_score(
            survival_train=y_train,
            survival_test=y_test,
            estimate=S_test,
            times=times_eval,
        )
        n_times = len(times_brier)
        S_test_valid = S_test[:, :n_times]

        # Baseline IBS
        ibs_base = integrated_brier_score(
            survival_train=y_train,
            survival_test=y_test,
            estimate=S_test_valid,
            times=times_brier,
        )

        # Permutation importance - optimized loop
        fold_drops = np.zeros((n_features, n_repeats))
        X_test_perm = X_test.copy()

        for col_idx in range(n_features):
            col_original = X_test[:, col_idx].copy()

            for r in range(n_repeats):
                # In-place permutation
                X_test_perm[:, col_idx] = local_rng.permutation(col_original)

                # Fast prediction using risk scores
                S_perm = get_survival_matrix(X_test_perm)
                S_perm_valid = S_perm[:, :n_times]

                ibs_perm = integrated_brier_score(
                    survival_train=y_train,
                    survival_test=y_test,
                    estimate=S_perm_valid,
                    times=times_brier,
                )
                fold_drops[col_idx, r] = ibs_perm - ibs_base

            # Restore original
            X_test_perm[:, col_idx] = col_original

        return ibs_base, fold_drops

    print(f"\n=== {n_imputations} imputations – {n_splits}-fold CV permutation importance (IBS) ===")
    print(f"Total iterations: {n_imputations * n_splits}")

    # Parallelize with progress
    results = Parallel(n_jobs=n_jobs, backend='loky', verbose=10)(
        delayed(compute_fold)(d, fold_idx, train_idx, test_idx)
        for d in range(n_imputations)
        for fold_idx, (train_idx, test_idx) in enumerate(cv_splits)
    )

    # Collect results efficiently
    baseline_ibs_list = [res[0] for res in results]

    # Aggregate drops using numpy for speed
    all_drops = np.concatenate([res[1] for res in results], axis=1)  # shape: (n_features, total_repeats)

    # Aggregate feature importances
    df_imp_ibs = pd.DataFrame({
        "feature": feature_names,
        "mean_increase_ibs": all_drops.mean(axis=1),
        "sd_increase_ibs": all_drops.std(axis=1, ddof=1),
        "n_evals": all_drops.shape[1],
    }).sort_values("mean_increase_ibs", ascending=False).reset_index(drop=True)

    # Baseline IBS summary
    baseline_ibs_mean = np.mean(baseline_ibs_list)
    baseline_ibs_sd = np.std(baseline_ibs_list, ddof=1) if len(baseline_ibs_list) > 1 else 0.0

    print("\n=== Baseline CV IBS over imputations & folds ===")
    print(f"Mean ± SD: {baseline_ibs_mean:.4f} ± {baseline_ibs_sd:.4f}")

    return baseline_ibs_mean, baseline_ibs_sd, df_imp_ibs

### Execute
- **`X_list`** (set to `X_reduced_list` in your code): This is a list of Pandas DataFrames. The function loops over each imputation separately during CV, fitting models and computing importances for robustness.

- **`y_surv`** (set to `y_surv_readm`): This is your survival outcome/target variable, as a NumPy structured array with two fields: 'event' (bool: True if event like death/readmission occurred, False if censored) and 'time' (float: time to event or censoring).

- **`times_eval`** (set to `times_eval_readm`): This is a NumPy array or list of evenly spaced time points covering your data's range. More points = finer evaluation but slightly higher computation. Too few = coarse/less accurate IBS.

- **`alpha_idx`** (set to 25, default=25): This is the index in the Coxnet model's alpha path (regularization strengths) to use for predictions and importance. Selects the effective alpha for the model after fitting (e.g., index 0 is strongest regularization, higher indices are weaker). Typical values range 0-50. Lower index = more regularization (a sparser model with fewer features used); Higher = less regularization (more complex model). Tune via separate CV if needed.

- **`n_splits`** (set to 5, default=5):  This is the number of folds in K-Fold CV.  Splits data into train/test sets repeatedly (e.g., 5 folds = 80% train, 20% test per fold). Precomputed once and reused across imputations.  Balances bias-variance; 5 is a good default for moderate datasets.

- **`n_repeats`** (set to 5, default=3): This is the number of times to repeat the permutation for each feature in each fold/imputation. For each feature, shuffles its test-set values `n_repeats` times, recomputes IBS each time, and averages the drops for stability. Increase for better precision and more reliable importance scores (lower SD in output DF), but runtime increases linearly (e.g., doubles if going from 3 to 6).

### Default Arguments (Not Specified in Your Call, So They Use These Values)
These are less commonly changed but control the underlying Coxnet model and computation:
- `random_state=2125`: Seed for CV splitting and permutations (ensures reproducibility).
- `l1_ratio=0.9`: Elastic net mixing (0.9 = mostly L1/Lasso for feature selection, 0 = L2/Ridge).
- `alpha_min_ratio=0.01`: Smallest alpha as a fraction of the max (controls regularization range).
- `max_iter=100000`: Max iterations for model convergence (high to avoid early stopping).
- `n_jobs=-1`: Parallel jobs (uses all CPU cores for speed via Joblib).

In [9]:
baseline_ibs_death, baseline_ibs_sd_death, df_imp_ibs_death = (
    permutation_importance_ibs_cv_mi(
        X_list=X_reduced_list,
        y_surv=y_surv_death,
        times_eval=times_eval_death,
        alpha_idx=25,
        n_splits=5,
        n_repeats=5,
        n_jobs=-1
    )
)
# Top 20 predictors for death
df_imp_ibs_death.head(20)


=== 5 imputations – 5-fold CV permutation importance (IBS) ===
Total iterations: 25


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:   49.6s
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done  13 out of  25 | elapsed:  1.7min remaining:  1.5min
[Parallel(n_jobs=-1)]: Done  16 out of  25 | elapsed:  1.7min remaining:   56.6s
[Parallel(n_jobs=-1)]: Done  19 out of  25 | elapsed:  2.4min remaining:   46.3s
[Parallel(n_jobs=-1)]: Done  22 out of  25 | elapsed:  2.5min remaining:   20.2s



=== Baseline CV IBS over imputations & folds ===
Mean ± SD: 0.0351 ± 0.0006


[Parallel(n_jobs=-1)]: Done  25 out of  25 | elapsed:  2.9min finished


,feature,mean_increase_ibs,sd_increase_ibs,n_evals
0,feature_17,0.000003,0.000002,125
1,feature_1,0.000000,0.000000,125
2,feature_0,0.000000,0.000000,125
3,feature_3,0.000000,0.000000,125
4,feature_4,0.000000,0.000000,125
5,feature_5,0.000000,0.000000,125
6,feature_2,0.000000,0.000000,125
7,feature_7,0.000000,0.000000,125
8,feature_8,0.000000,0.000000,125
9,feature_9,0.000000,0.000000,125


In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import brier_score, integrated_brier_score
from joblib import Parallel, delayed

def permutation_importance_ibs_cv_mi2(
    X_list,
    y_surv,
    times_eval,
    alpha_idx=25,
    n_splits=5,
    n_repeats=3,
    random_state=2125,
    l1_ratio=0.9,
    alpha_min_ratio=0.01,
    n_alphas=50,
    max_iter=100000,
    n_jobs=-1,
):
    """
    Highly optimized version using risk scores and coefficient-based updates.

    Parameters
    ----------
    X_list : list of pandas.DataFrame
        List of imputed datasets (same samples, different imputations)
    y_surv : structured array
        Survival outcome (time, event)
    times_eval : array-like
        Time points for evaluation
    alpha_idx : int
        Index of alpha to use from regularization path
    n_splits : int
        Number of CV folds
    n_repeats : int
        Number of permutation repeats per feature
    random_state : int
        Random seed for reproducibility
    l1_ratio : float
        ElasticNet mixing parameter (0 <= l1_ratio <= 1)
    alpha_min_ratio : float
        Minimum alpha as fraction of maximum alpha
    n_alphas : int
        Number of alphas along regularization path
    max_iter : int
        Maximum iterations for model fitting
    n_jobs : int
        Number of parallel jobs (-1 for all cores)

    Returns
    -------
    baseline_ibs_mean : float
        Mean baseline IBS across folds and imputations
    baseline_ibs_sd : float
        Standard deviation of baseline IBS
    df_imp_ibs : pandas.DataFrame
        Dataframe with permutation importance results
    """
    # Convert to NumPy arrays upfront
    feature_names = X_list[0].columns.tolist()
    n_features = len(feature_names)
    X_list = [X.values.astype(np.float64, copy=False) for X in X_list]
    times_eval = np.asarray(times_eval, dtype=np.float64)
    n_imputations = len(X_list)

    # Precompute CV splits once
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cv_splits = list(kf.split(np.arange(X_list[0].shape[0])))

    def compute_fold_optimized(d, fold_idx, train_idx, test_idx):
        """
        Optimized fold computation using coefficient-based risk updates.
        """
        X_imp = X_list[d]
        X_train = X_imp[train_idx]
        X_test = X_imp[test_idx]
        y_train = y_surv[train_idx]
        y_test = y_surv[test_idx]

        local_rng = np.random.RandomState(random_state + d * n_splits + fold_idx)

        # Fit Coxnet
        model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            alpha_min_ratio=alpha_min_ratio,
            n_alphas=n_alphas,
            normalize=True,
            fit_baseline_model=True,
            max_iter=max_iter,
            verbose=False,
        )
        model.fit(X_train, y_train)

        # Handle alpha selection safely
        eff_alpha_idx = min(alpha_idx, len(model.alphas_) - 1)
        alpha_val = model.alphas_[eff_alpha_idx]

        # Extract coefficients for risk score computation
        coef = model.coef_[:, eff_alpha_idx]

        # Access baseline survival
        baseline_estimator = model._baseline_models[eff_alpha_idx]
        S0 = baseline_estimator.baseline_survival_(times_eval)

        # Precompute baseline risk scores
        baseline_risks = model.predict(X_test, alpha=alpha_val)

        # Compute baseline survival probabilities
        S_test = np.power(S0[None, :], np.exp(baseline_risks[:, None]))

        # Get effective time grid for Brier score
        times_brier, _ = brier_score(
            survival_train=y_train,
            survival_test=y_test,
            estimate=S_test,
            times=times_eval,
        )
        n_times = len(times_brier)
        S_test_valid = S_test[:, :n_times]

        # Baseline IBS
        ibs_base = integrated_brier_score(
            survival_train=y_train,
            survival_test=y_test,
            estimate=S_test_valid,
            times=times_brier,
        )

        # Optimized permutation importance using risk score updates
        fold_drops = np.zeros((n_features, n_repeats))

        for col_idx in range(n_features):
            col_original = X_test[:, col_idx].copy()
            coef_col = coef[col_idx]

            for r in range(n_repeats):
                # Generate permutation
                permuted_values = local_rng.permutation(col_original)

                # Fast risk score update instead of full prediction
                risk_delta = (permuted_values - col_original) * coef_col
                risks_perm = baseline_risks + risk_delta

                # Compute survival probabilities from updated risks
                S_perm = np.power(S0[None, :], np.exp(risks_perm[:, None]))
                S_perm_valid = S_perm[:, :n_times]

                # Compute IBS for permuted feature
                ibs_perm = integrated_brier_score(
                    survival_train=y_train,
                    survival_test=y_test,
                    estimate=S_perm_valid,
                    times=times_brier,
                )
                fold_drops[col_idx, r] = ibs_perm - ibs_base

        return ibs_base, fold_drops

    print(f"\n=== {n_imputations} imputations – {n_splits}-fold CV permutation importance (IBS) ===")
    print(f"Total iterations: {n_imputations * n_splits}")
    print(f"Features: {n_features}, Permutation repeats: {n_repeats}")

    # Parallel execution
    results = Parallel(n_jobs=n_jobs, backend='loky', verbose=10)(
        delayed(compute_fold_optimized)(d, fold_idx, train_idx, test_idx)
        for d in range(n_imputations)
        for fold_idx, (train_idx, test_idx) in enumerate(cv_splits)
    )

    # Aggregate results
    baseline_ibs_list = [res[0] for res in results]

    # Concatenate all permutation results
    all_drops = np.concatenate([res[1] for res in results], axis=1)

    # Create results dataframe
    df_imp_ibs = pd.DataFrame({
        "feature": feature_names,
        "mean_increase_ibs": all_drops.mean(axis=1),
        "sd_increase_ibs": all_drops.std(axis=1, ddof=1),
        "n_evals": all_drops.shape[1],
    }).sort_values("mean_increase_ibs", ascending=False).reset_index(drop=True)

    # Baseline IBS statistics
    baseline_ibs_mean = np.mean(baseline_ibs_list)
    baseline_ibs_sd = np.std(baseline_ibs_list, ddof=1) if len(baseline_ibs_list) > 1 else 0.0

    print("\n=== Baseline CV IBS over imputations & folds ===")
    print(f"Mean ± SD: {baseline_ibs_mean:.4f} ± {baseline_ibs_sd:.4f}")
    print(f"Range: [{np.min(baseline_ibs_list):.4f}, {np.max(baseline_ibs_list):.4f}]")

    print("\n=== Top 20 Most Important Features ===")
    print(df_imp_ibs.head(20).to_string(index=False))

    return baseline_ibs_mean, baseline_ibs_sd, df_imp_ibs

In [11]:
baseline_ibs_readm2, baseline_ibs_sd_readm2, df_imp_ibs_readm2 = (
    permutation_importance_ibs_cv_mi2(
        X_list=X_reduced_list,
        y_surv=y_surv_readm,
        times_eval=times_eval_readm,
        alpha_idx=25,
        n_splits=10,
        n_repeats=10,
        n_jobs=-1
    )
)


=== 5 imputations – 10-fold CV permutation importance (IBS) ===
Total iterations: 50
Features: 79, Permutation repeats: 10


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:  3.3min
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:  3.3min
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:  6.5min
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  8.2min
[Parallel(n_jobs=-1)]: Done  41 out of  50 | elapsed:  9.7min remaining:  2.1min
[Parallel(n_jobs=-1)]: Done  47 out of  50 | elapsed:  9.8min remaining:   37.7s



=== Baseline CV IBS over imputations & folds ===
Mean ± SD: 0.1427 ± 0.0015
Range: [0.1409, 0.1455]

=== Top 20 Most Important Features ===
   feature  mean_increase_ibs  sd_increase_ibs  n_evals
feature_15       3.761013e-05         0.000043      500
feature_49       2.501684e-05         0.000037      500
feature_50       1.492352e-05         0.000022      500
 feature_0       9.436245e-06         0.000016      500
feature_12       8.349818e-06         0.000019      500
feature_57       8.023716e-06         0.000022      500
feature_14       7.226417e-06         0.000016      500
feature_28       6.105699e-06         0.000030      500
 feature_1       4.009617e-06         0.000016      500
feature_21       3.737994e-06         0.000016      500
feature_23       3.603043e-06         0.000017      500
feature_64       3.367812e-06         0.000021      500
feature_17       2.278776e-06         0.000010      500
feature_16       2.127283e-06         0.000014      500
feature_75       2.

[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed: 10.7min finished


In [12]:
# Top 20 predictors for death
df_imp_ibs_readm2.head(20)

,feature,mean_increase_ibs,sd_increase_ibs,n_evals
0,feature_15,3.761013e-05,0.000043,500
1,feature_49,2.501684e-05,0.000037,500
2,feature_50,1.492352e-05,0.000022,500
3,feature_0,9.436245e-06,0.000016,500
4,feature_12,8.349818e-06,0.000019,500
5,feature_57,8.023716e-06,0.000022,500
6,feature_14,7.226417e-06,0.000016,500
7,feature_28,6.105699e-06,0.000030,500
8,feature_1,4.009617e-06,0.000016,500
9,feature_21,3.737994e-06,0.000016,500


In [ ]:
baseline_ibs_death2, baseline_ibs_sd_death2, df_imp_ibs_death2 = (
    permutation_importance_ibs_cv_mi2(
        X_list=X_reduced_list,
        y_surv=y_surv_death,
        times_eval=times_eval_death,
        alpha_idx=25,
        n_splits=10,
        n_repeats=10,
        n_jobs=-1
    )
)
# Top 20 predictors for death
df_imp_ibs_death2.head(20)


=== 5 imputations – 10-fold CV permutation importance (IBS) ===
Total iterations: 50
Features: 79, Permutation repeats: 10


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:  2.8min
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:  2.8min
[Parallel(n_jobs=-1)]: Done  25 tasks      | elapsed:  5.5min
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:  6.9min
[Parallel(n_jobs=-1)]: Done  41 out of  50 | elapsed:  8.3min remaining:  1.8min
[Parallel(n_jobs=-1)]: Done  47 out of  50 | elapsed:  8.3min remaining:   31.9s



=== Baseline CV IBS over imputations & folds ===
Mean ± SD: 0.0351 ± 0.0010
Range: [0.0340, 0.0372]

=== Top 20 Most Important Features ===
   feature  mean_increase_ibs  sd_increase_ibs  n_evals
feature_17           0.000004         0.000003      500
 feature_1           0.000000         0.000000      500
 feature_0           0.000000         0.000000      500
 feature_3           0.000000         0.000000      500
 feature_4           0.000000         0.000000      500
 feature_5           0.000000         0.000000      500
 feature_2           0.000000         0.000000      500
 feature_7           0.000000         0.000000      500
 feature_8           0.000000         0.000000      500
 feature_9           0.000000         0.000000      500
feature_10           0.000000         0.000000      500
feature_11           0.000000         0.000000      500
feature_12           0.000000         0.000000      500
feature_13           0.000000         0.000000      500
 feature_6         

[Parallel(n_jobs=-1)]: Done  50 out of  50 | elapsed:  9.1min finished


,feature,mean_increase_ibs,sd_increase_ibs,n_evals
0,feature_17,0.000004,0.000003,500
1,feature_1,0.000000,0.000000,500
2,feature_0,0.000000,0.000000,500
3,feature_3,0.000000,0.000000,500
4,feature_4,0.000000,0.000000,500
5,feature_5,0.000000,0.000000,500
6,feature_2,0.000000,0.000000,500
7,feature_7,0.000000,0.000000,500
8,feature_8,0.000000,0.000000,500
9,feature_9,0.000000,0.000000,500


In [ ]:
df_imp_ibs_death2.head(20)

,feature,mean_increase_ibs,sd_increase_ibs,n_evals
0,feature_17,0.000004,0.000003,500
1,feature_1,0.000000,0.000000,500
2,feature_0,0.000000,0.000000,500
3,feature_3,0.000000,0.000000,500
4,feature_4,0.000000,0.000000,500
5,feature_5,0.000000,0.000000,500
6,feature_2,0.000000,0.000000,500
7,feature_7,0.000000,0.000000,500
8,feature_8,0.000000,0.000000,500
9,feature_9,0.000000,0.000000,500
